In [1]:
import os
import glob
import numpy as np
import pandas as pd

ROOT = r'D:\Bluestock work\bluestock-fintech-analysis1'

RAW_DIR = os.path.join(ROOT, 'data', 'raw')
PROCESSED_DIR = os.path.join(ROOT, 'data', 'processed')
REPORTS_DIR = os.path.join(ROOT, 'reports')

paths = glob.glob(os.path.join(RAW_DIR, '*08_investor_transactions*'))
print(paths)

['D:\\Bluestock work\\bluestock-fintech-analysis1\\data\\raw\\1788499980509-304c1255-08_investor_transactions.csv']


In [2]:
tx = pd.read_csv(paths[0])
n_start = len(tx)

tx['transaction_type_raw'] = tx['transaction_type']
tx['kyc_status_raw'] = tx['kyc_status']

print(tx.shape)
print(tx.dtypes)
print(tx.isnull().sum())

(32778, 15)
investor_id                 str
transaction_date            str
amfi_code                 int64
transaction_type            str
amount_inr                int64
state                       str
city                        str
city_tier                   str
age_group                   str
gender                      str
annual_income_lakh      float64
payment_mode                str
kyc_status                  str
transaction_type_raw        str
kyc_status_raw              str
dtype: object
investor_id             0
transaction_date        0
amfi_code               0
transaction_type        0
amount_inr              0
state                   0
city                    0
city_tier               0
age_group               0
gender                  0
annual_income_lakh      0
payment_mode            0
kyc_status              0
transaction_type_raw    0
kyc_status_raw          0
dtype: int64


In [3]:
log = []

def add_log(step, df):
    log.append({'step': step, 'rows': len(df)})

add_log('start', tx)

In [4]:
n_exact_dups = tx.duplicated().sum()
print('Rows that are identical to another row:', n_exact_dups)

Rows that are identical to another row: 0


In [5]:
print(tx['transaction_type_raw'].value_counts(dropna=False))

type_key = tx['transaction_type_raw'].str.lower().str.replace(r'[^a-z]', '', regex=True)
print(type_key.value_counts(dropna=False))

transaction_type_raw
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64
transaction_type_raw
sip           19716
lumpsum        8095
redemption     4967
Name: count, dtype: int64


In [6]:
TYPE_MAP = {
    'sip': 'SIP',
    'lumpsum': 'Lumpsum',
    'redemption': 'Redemption',
}

tx['transaction_type'] = type_key.map(TYPE_MAP)

unmapped = tx['transaction_type'].isnull() & type_key.notnull()
print('Values not in the map:', unmapped.sum())
print(type_key[unmapped].value_counts())

Values not in the map: 0
Series([], Name: count, dtype: int64)


In [7]:
print('Data type:', tx['amount_inr'].dtype)

amount_missing_before = tx['amount_inr'].isnull().sum()
amount = pd.to_numeric(tx['amount_inr'], errors='coerce')
n_amount_not_number = amount.isnull().sum() - amount_missing_before

print('Missing amounts in the raw file:', amount_missing_before)
print('Amounts that are not numbers:', n_amount_not_number)

not_number = amount.isnull() & tx['amount_inr'].notnull()
print(tx.loc[not_number, 'amount_inr'].head(10))

Data type: int64
Missing amounts in the raw file: 0
Amounts that are not numbers: 0
Series([], Name: amount_inr, dtype: int64)


In [8]:
print('Amount zero:', (tx['amount_inr'] == 0).sum())
print('Amount negative:', (tx['amount_inr'] < 0).sum())

print(tx.groupby('transaction_type')['amount_inr'].describe().round(0))

Amount zero: 0
Amount negative: 0
                    count      mean       std     min       25%       50%  \
transaction_type                                                            
Lumpsum            8095.0  254456.0  145977.0  4541.0  129928.0  251770.0   
Redemption         4967.0  250559.0  145485.0  4550.0  126554.0  247511.0   
SIP               19716.0   11018.0   14218.0   400.0    2117.0    5139.0   

                       75%       max  
transaction_type                      
Lumpsum           373162.0  594649.0  
Redemption        365979.0  597498.0  
SIP                13554.0   59998.0  


In [9]:
raw_clean = tx['transaction_date'].astype(str).str.strip()
raw_clean = raw_clean.where(tx['transaction_date'].notnull())

shapes = (raw_clean.str.replace(r'\d', '9', regex=True)
                   .str.replace(r'[A-Za-z]', 'a', regex=True))

format_counts = shapes.value_counts(dropna=False)
print(format_counts)

transaction_date
9999-99-99    32778
Name: count, dtype: int64


In [10]:
print(tx['kyc_status_raw'].value_counts(dropna=False))

kyc_key = tx['kyc_status_raw'].str.lower().str.replace(r'[^a-z]', '', regex=True)
print(kyc_key.value_counts(dropna=False))

kyc_status_raw
Verified    30146
Pending      2632
Name: count, dtype: int64
kyc_status_raw
verified    30146
pending      2632
Name: count, dtype: int64


In [11]:
assert tx['transaction_type'].isin(['SIP', 'Lumpsum', 'Redemption']).all(), 'Unknown transaction type'
assert (tx['amount_inr'] > 0).all(), 'Amount is not above 0'
assert tx['transaction_date'].notnull().all(), 'Missing date'
assert str(tx['transaction_date'].dtype).startswith('datetime64'), 'Date is not a datetime'
assert tx['kyc_status'].dropna().isin(set(KYC_MAP.values())).all(), 'KYC value not allowed'

print('All checks passed')

AssertionError: Date is not a datetime

In [12]:
raw_clean = tx['transaction_date'].astype(str).str.strip()
raw_clean = raw_clean.where(tx['transaction_date'].notnull())

shapes = (raw_clean.str.replace(r'\d', '9', regex=True)
                   .str.replace(r'[A-Za-z]', 'a', regex=True))

format_counts = shapes.value_counts(dropna=False)
print(format_counts)

transaction_date
9999-99-99    32778
Name: count, dtype: int64


In [14]:
date_formats = {
    '9999-99-99': '%Y-%m-%d',
    '99/99/9999': '%d/%m/%Y',
    '99-aaa-9999': '%d-%b-%Y',
}

parsed = pd.Series(pd.NaT, index=tx.index)

for shape, fmt in date_formats.items():
    attempt = pd.to_datetime(raw_clean.where(shapes == shape), format=fmt, errors='coerce')
    parsed = parsed.fillna(attempt)

not_read = parsed.isnull() & raw_clean.notnull()
print('Dates not read:', not_read.sum())
print(shapes[not_read].value_counts())

Dates not read: 0
Series([], Name: count, dtype: int64)


In [15]:
tx['transaction_date'] = parsed.dt.normalize()

bad_date_rows = tx[tx['transaction_date'].isnull()]
n_bad_date = len(bad_date_rows)

tx = tx[tx['transaction_date'].notnull()]
add_log('dates parsed, unreadable dates removed', tx)

today = pd.Timestamp.today().normalize()
print('Removed:', n_bad_date)
print('Oldest date:', tx['transaction_date'].min())
print('Newest date:', tx['transaction_date'].max())
print('Dates in the future:', (tx['transaction_date'] > today).sum())

Removed: 0
Oldest date: 2024-01-01 00:00:00
Newest date: 2025-05-30 00:00:00
Dates in the future: 0


In [16]:
print(tx['kyc_status_raw'].value_counts(dropna=False))

kyc_key = tx['kyc_status_raw'].str.lower().str.replace(r'[^a-z]', '', regex=True)
print(kyc_key.value_counts(dropna=False))

kyc_status_raw
Verified    30146
Pending      2632
Name: count, dtype: int64
kyc_status_raw
verified    30146
pending      2632
Name: count, dtype: int64


In [17]:
KYC_MAP = {
    'verified': 'Verified',
    'pending': 'Pending',
    'rejected': 'Rejected',
}

tx['kyc_status'] = kyc_key.map(KYC_MAP)

n_kyc_missing = tx['kyc_status_raw'].isnull().sum()
kyc_not_allowed = tx['kyc_status'].isnull() & kyc_key.notnull()
n_kyc_not_allowed = kyc_not_allowed.sum()

print('KYC missing:', n_kyc_missing)
print('KYC value not in the allowed list:', n_kyc_not_allowed)
print(kyc_key[kyc_not_allowed].value_counts())

KYC missing: 0
KYC value not in the allowed list: 0
Series([], Name: count, dtype: int64)


In [18]:
pd.crosstab(tx['transaction_type'], tx['kyc_status'])

kyc_status,Pending,Verified
transaction_type,,
Lumpsum,669,7426
Redemption,378,4589
SIP,1585,18131


In [19]:
assert tx['transaction_type'].isin(['SIP', 'Lumpsum', 'Redemption']).all(), 'Unknown transaction type'
assert (tx['amount_inr'] > 0).all(), 'Amount is not above 0'
assert tx['transaction_date'].notnull().all(), 'Missing date'
assert str(tx['transaction_date'].dtype).startswith('datetime64'), 'Date is not a datetime'
assert tx['kyc_status'].dropna().isin(set(KYC_MAP.values())).all(), 'KYC value not allowed'

print('All checks passed')

All checks passed


In [23]:
clean = tx.drop(columns=['transaction_type_raw', 'kyc_status_raw'])
clean.to_csv(os.path.join(PROCESSED_DIR, 'investor_transactions_clean.csv'), index=False)



check = pd.read_csv(os.path.join(PROCESSED_DIR, 'investor_transactions_clean.csv'), parse_dates=['transaction_date'])
print(check.shape)
print(check.dtypes)

(32778, 13)
investor_id                      str
transaction_date      datetime64[us]
amfi_code                      int64
transaction_type                 str
amount_inr                     int64
state                            str
city                             str
city_tier                        str
age_group                        str
gender                           str
annual_income_lakh           float64
payment_mode                     str
kyc_status                       str
dtype: object


In [26]:
lines = [
    '# Investor transactions cleaning summary',
    '',
    f'- Rows at start: {n_start}. Rows at end: {len(clean)}.',
    f'- Identical rows (not removed, no transaction ID): {n_exact_dups}.',
    f'- Rows for each type: {tx["transaction_type"].value_counts().to_dict()}.',
    f'- Date formats found: {format_counts.to_dict()}.',
    f'- Rows removed (date missing or not readable): {n_bad_date}.',
    f'- KYC status: {tx["kyc_status"].value_counts(dropna=False).to_dict()}.',
    f'- KYC missing: {n_kyc_missing}. KYC value not in the allowed list: {n_kyc_not_allowed}.',
    '- Final checks passed: types, amounts, dates, KYC values.',
    '',
    '## Assumptions to confirm with my manager',
    '- Identical rows can be real transactions, so they were kept.',
    '- All amounts must be positive. The transaction type shows the direction.',
    '- The list of allowed KYC values.',
]

text = '\n'.join(lines)

with open(os.path.join(REPORTS_DIR, 'investor_transactions_cleaning_summary.md'), 'w', encoding='utf-8') as f:
    f.write(text)

print(text)

# Investor transactions cleaning summary

- Rows at start: 32778. Rows at end: 32778.
- Identical rows (not removed, no transaction ID): 0.
- Rows for each type: {'SIP': 19716, 'Lumpsum': 8095, 'Redemption': 4967}.
- Date formats found: {'9999-99-99': 32778}.
- Rows removed (date missing or not readable): 0.
- KYC status: {'Verified': 30146, 'Pending': 2632}.
- KYC missing: 0. KYC value not in the allowed list: 0.
- Final checks passed: types, amounts, dates, KYC values.

## Assumptions to confirm with my manager
- Identical rows can be real transactions, so they were kept.
- All amounts must be positive. The transaction type shows the direction.
- The list of allowed KYC values.
